In [ ]:
# Import required libraries
import sys
sys.path.append('..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
%matplotlib inline

from src.extractor import MidiNote, midi_number_to_note_name
from src.optimizer import FretboardOptimizer, FretPosition

## 1. Create Test Melody

In [ ]:
# Simple test melody: C-E-G (C major chord arpeggio)
melody = [
    MidiNote(60, 0.0, 0.5),  # C4
    MidiNote(64, 0.5, 1.0),  # E4
    MidiNote(67, 1.0, 1.5),  # G4
]

print("Test Melody:")
for i, note in enumerate(melody):
    print(f"{i+1}. {midi_number_to_note_name(note.midi_number)} (MIDI {note.midi_number})")

## 2. Build Graph and Find Path

In [ ]:
# Create optimizer
optimizer = FretboardOptimizer()

# Build graph
optimizer.build_graph(melody)

print(f"Graph built with {len(optimizer.graph)} nodes")
print(f"\nPositions per note:")
for note_idx, positions in optimizer.positions_by_note.items():
    note_name = midi_number_to_note_name(melody[note_idx].midi_number)
    print(f"  {note_name}: {len(positions)} positions")

# Find optimal path
path = optimizer.find_best_path()

print(f"\nOptimal path found with {len(path)} positions:")
string_names = ['E (low)', 'A', 'D', 'G', 'B', 'E (high)']
for i, pos in enumerate(path):
    note_name = midi_number_to_note_name(melody[i].midi_number)
    print(f"  {note_name}: String {pos.string + 1} ({string_names[pos.string]}), Fret {pos.fret}")

## 3. Visualize Fretboard with All Possible Positions

In [ ]:
def draw_fretboard(ax, num_frets=12, num_strings=6):
    """Draw a guitar fretboard."""
    # Draw frets (vertical lines)
    for fret in range(num_frets + 1):
        ax.axvline(fret, color='black', linewidth=1 if fret > 0 else 3)
    
    # Draw strings (horizontal lines)
    for string in range(num_strings):
        ax.axhline(string, color='gray', linewidth=0.5)
    
    # Add fret markers
    markers = [3, 5, 7, 9, 12]
    for marker in markers:
        if marker <= num_frets:
            ax.plot(marker - 0.5, num_strings / 2 - 0.5, 'o', 
                   color='lightgray', markersize=10, alpha=0.5)
    
    ax.set_xlim(-0.5, num_frets + 0.5)
    ax.set_ylim(-0.5, num_strings - 0.5)
    ax.set_aspect('equal')
    ax.invert_yaxis()  # High E on top
    
    # Labels
    string_names = ['E', 'B', 'G', 'D', 'A', 'E']
    ax.set_yticks(range(num_strings))
    ax.set_yticklabels(string_names)
    ax.set_xlabel('Fret')
    ax.set_title('Guitar Fretboard')

# Create figure
fig, ax = plt.subplots(figsize=(16, 6))
draw_fretboard(ax)

# Plot all possible positions for each note
colors = ['red', 'green', 'blue', 'orange', 'purple', 'brown']

for note_idx, positions in optimizer.positions_by_note.items():
    note_name = midi_number_to_note_name(melody[note_idx].midi_number)
    color = colors[note_idx % len(colors)]
    
    for pos in positions:
        # Convert to display coordinates
        string_display = 5 - pos.string  # Invert for display
        
        # Draw circle for position
        circle = plt.Circle((pos.fret - 0.5, string_display), 0.3, 
                          color=color, alpha=0.3, label=note_name if pos == positions[0] else '')
        ax.add_patch(circle)

ax.legend(loc='upper right')
plt.title('All Possible Positions for Each Note')
plt.tight_layout()
plt.show()

## 4. Visualize Optimal Path

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
draw_fretboard(ax)

# Plot all possible positions (faded)
for note_idx, positions in optimizer.positions_by_note.items():
    for pos in positions:
        string_display = 5 - pos.string
        circle = plt.Circle((pos.fret - 0.5, string_display), 0.25, 
                          color='lightgray', alpha=0.2)
        ax.add_patch(circle)

# Highlight optimal path
for i, pos in enumerate(path):
    string_display = 5 - pos.string
    note_name = midi_number_to_note_name(melody[i].midi_number)
    
    # Draw circle for chosen position
    circle = plt.Circle((pos.fret - 0.5, string_display), 0.35, 
                      color='green', alpha=0.8)
    ax.add_patch(circle)
    
    # Add note label
    ax.text(pos.fret - 0.5, string_display, note_name, 
           ha='center', va='center', fontsize=10, weight='bold')
    
    # Draw arrow to next position
    if i < len(path) - 1:
        next_pos = path[i + 1]
        next_string_display = 5 - next_pos.string
        
        ax.annotate('', xy=(next_pos.fret - 0.5, next_string_display),
                   xytext=(pos.fret - 0.5, string_display),
                   arrowprops=dict(arrowstyle='->', lw=2, color='red', alpha=0.6))

plt.title('Optimal Fingering Path (Dijkstra\'s Algorithm)', fontsize=14, weight='bold')
plt.tight_layout()
plt.show()

## 5. Analyze Transition Costs

In [ ]:
print("Transition Cost Analysis:")
print("=" * 80)

total_cost = 0

for i in range(len(path) - 1):
    pos1 = path[i]
    pos2 = path[i + 1]
    
    cost = optimizer.calculate_transition_cost(pos1, pos2)
    total_cost += cost
    
    fret_dist = abs(pos1.fret - pos2.fret)
    string_jump = abs(pos1.string - pos2.string)
    
    note1 = midi_number_to_note_name(melody[i].midi_number)
    note2 = midi_number_to_note_name(melody[i + 1].midi_number)
    
    print(f"\nTransition {i+1}: {note1} → {note2}")
    print(f"  From: String {pos1.string + 1}, Fret {pos1.fret}")
    print(f"  To:   String {pos2.string + 1}, Fret {pos2.fret}")
    print(f"  Fret distance:  {fret_dist}")
    print(f"  String jump:    {string_jump}")
    print(f"  Transition cost: {cost:.2f}")

print(f"\n{'='*80}")
print(f"Total path cost: {total_cost:.2f}")

## 6. Compare Different Cost Configurations

In [ ]:
# Test different configurations
configs = [
    {"name": "Default", "string_weight": 0.5},
    {"name": "Prefer Same String", "string_weight": 0.1},
    {"name": "Prefer Different Strings", "string_weight": 2.0},
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, config in enumerate(configs):
    ax = axes[idx]
    
    # Create optimizer with custom config
    opt = FretboardOptimizer()
    opt.STRING_JUMP_WEIGHT = config["string_weight"]
    
    # Optimize
    path_config = opt.optimize(melody)
    
    # Draw fretboard
    draw_fretboard(ax, num_frets=12)
    
    # Plot path
    for i, pos in enumerate(path_config):
        string_display = 5 - pos.string
        note_name = midi_number_to_note_name(melody[i].midi_number)
        
        circle = plt.Circle((pos.fret - 0.5, string_display), 0.35, 
                          color='green', alpha=0.8)
        ax.add_patch(circle)
        ax.text(pos.fret - 0.5, string_display, note_name, 
               ha='center', va='center', fontsize=9, weight='bold')
        
        if i < len(path_config) - 1:
            next_pos = path_config[i + 1]
            next_string_display = 5 - next_pos.string
            ax.annotate('', xy=(next_pos.fret - 0.5, next_string_display),
                       xytext=(pos.fret - 0.5, string_display),
                       arrowprops=dict(arrowstyle='->', lw=2, color='red', alpha=0.6))
    
    ax.set_title(f"{config['name']}\n(String Weight: {config['string_weight']})")

plt.tight_layout()
plt.show()

## 7. Graph Structure Visualization

In [ ]:
# Count nodes and edges
num_nodes = len([n for n in optimizer.graph if n != ('START', None) and n != ('END', None)])
num_edges = sum(len(neighbors) for node, neighbors in optimizer.graph.items() 
               if node != ('START', None) and node != ('END', None))

print("Graph Statistics:")
print("=" * 60)
print(f"Number of notes:     {len(melody)}")
print(f"Number of nodes:     {num_nodes}")
print(f"Number of edges:     {num_edges}")
print(f"Average positions per note: {num_nodes / len(melody):.1f}")
print(f"Average edges per node:     {num_edges / num_nodes:.1f}")

# Visualize as network graph (simplified)
fig, ax = plt.subplots(figsize=(14, 8))

# Position nodes by note index and string
for note_idx, positions in optimizer.positions_by_note.items():
    for pos in positions:
        x = note_idx * 3
        y = (5 - pos.string) * 2 + pos.fret * 0.1
        
        # Highlight if in optimal path
        if pos in path:
            color = 'green'
            size = 300
            alpha = 1.0
        else:
            color = 'lightblue'
            size = 100
            alpha = 0.5
        
        ax.scatter(x, y, s=size, c=color, alpha=alpha, edgecolors='black')
        ax.text(x, y, f"{pos.string+1},{pos.fret}", 
               ha='center', va='center', fontsize=7)

# Draw edges for optimal path
for i in range(len(path) - 1):
    pos1 = path[i]
    pos2 = path[i + 1]
    
    x1 = i * 3
    y1 = (5 - pos1.string) * 2 + pos1.fret * 0.1
    x2 = (i + 1) * 3
    y2 = (5 - pos2.string) * 2 + pos2.fret * 0.1
    
    ax.plot([x1, x2], [y1, y2], 'r-', linewidth=2, alpha=0.6)

# Labels
for i, note in enumerate(melody):
    note_name = midi_number_to_note_name(note.midi_number)
    ax.text(i * 3, -2, note_name, ha='center', fontsize=12, weight='bold')

ax.set_xlabel('Note Index', fontsize=12)
ax.set_ylabel('Position (String, Fret)', fontsize=12)
ax.set_title('Graph Structure with Optimal Path (Green)', fontsize=14, weight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:
1. Building the graph structure for guitar fingering
2. Visualizing all possible positions on the fretboard
3. Showing the optimal path found by Dijkstra's algorithm
4. Analyzing transition costs
5. Comparing different cost configurations
6. Visualizing the graph as a network

**Key Insights:**
- The algorithm finds the path with minimum total cost
- Cost parameters can be tuned for different playing styles
- Visual representation helps understand the optimization